# 05 — Next-Week Demand Regression

## Notebook description

This notebook trains and compares models that predict next-week recorded sales for each eligible article-week. It begins with the persistence baseline from the original data-review notebook, then provides the structure for regularized linear regression, decision trees, Random Forest, and feasible boosting methods under chronological validation. Performance is measured primarily with MAE and RMSE, with R² reported as complementary context.

## Objectives

- Establish a leakage-safe persistence baseline on validation data.
- Build reproducible preprocessing and regression pipelines.
- Compare candidate models on later chronological windows.
- Report MAE, RMSE, and complementary R² with variation across windows.
- Inspect negative predictions, residual patterns, and segment-level errors.
- Select a candidate model without consulting the final test set.

## 1. Load the current modelling dataset

The detailed integrity and split audit is performed in `02_data_preparation.ipynb`. This notebook independently resolves the current complete publication so that its results are reproducible when run in isolation.

In [ ]:
from pathlib import Path
import json

import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error

ROOT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents]
     if (path / "src/data/process_data.py").is_file()
     and (path / "data/processed/CURRENT.json").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError(
        "Open fashion-demand-ml and run python src/data/process_data.py first."
    )

processed_dir = ROOT / "data/processed"
pointer = json.loads((processed_dir / "CURRENT.json").read_text(encoding="utf-8"))
run_dir = (processed_dir / pointer["run_dir"]).resolve()
if not run_dir.is_relative_to(processed_dir.resolve()) or pointer.get("preview"):
    raise ValueError("CURRENT.json does not reference a valid complete publication.")

model_data = pd.read_csv(
    run_dir / "article_week.csv.gz",
    dtype={"article_id": "string", "product_code": "string"},
    parse_dates=["week", "target_week"],
)
print("Run:", run_dir.name)
print("Rows and columns:", model_data.shape)

## 2. Persistence baseline

The baseline assumes that next week's recorded sales will equal the current week's recorded sales. It is evaluated on validation data only; final-test metrics remain hidden until the model and hyperparameters have been selected. MAE is expressed in recorded units per article-week, while RMSE places greater weight on large errors.

In [ ]:
validation = model_data.loc[model_data["split"] == "validation"]
observed_sales = validation["target_next_week_sales"]
baseline_predictions = validation["sales"]
baseline_metrics = pd.DataFrame([{
    "method": "Repeat current-week recorded sales",
    "partition": "validation",
    "MAE": mean_absolute_error(observed_sales, baseline_predictions),
    "RMSE": mean_squared_error(observed_sales, baseline_predictions) ** 0.5,
}])
display(baseline_metrics.round(3))

## 3. Planned modelling workflow

1. Define predictor groups and exclusions
2. Fit preprocessing using training data only
3. Train regularized linear and tree-based models
4. Compare models across chronological validation windows
5. Diagnose predictions and residuals
6. Analyse errors by demand level, product group, and period
7. Select the candidate model before opening the final test period

The target, split columns, observation flags, and identifiers must not be passed to the estimator as numeric predictors. Impossible negative predictions must be reported and handled with an explicit, consistently evaluated policy.

## Expected output

A baseline-relative comparison of regressors, diagnostic plots that follow the repository visualization rubric, a justified candidate model, and a critical account of remaining errors and operational constraints.